In [1]:
from transformers import AutoTokenizer, LlamaTokenizer, pipeline
import transformers
import torch
import numpy as np
from numpy import pad

In [2]:
!pip install 'accelerate>=0.26.0'

In [3]:
# the following code is based on this article :  https://medium.com/@sauquetalex/getting-word-embeddings-for-llama2-9a01c0d07d37

In [16]:
from langchain_ollama import OllamaLLM

model = "meta-llama/Llama-2-7b-chat-hf"
tokenizer = AutoTokenizer.from_pretrained(model)

tokenizer_config.json:   0%|          | 0.00/1.62k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

Some parameters are on the meta device because they were offloaded to the disk.


In [18]:
from sklearn.metrics.pairwise import cosine_similarity

pipeline = transformers.pipeline(
    "feature-extraction", model=model, torch_dtype=torch.float32, device_map="auto"
)

embeds = pipeline(
    [
        "soldier",
        "war",
    ],
    tokenizer=tokenizer,
    eos_token_id=tokenizer.eos_token_id,
)

# Checking if either of the strings is empty
if len(embeds[0][0]) <= 1 or len(embeds[1][0]) <= 1:
    print("Not enough tokens to compare")
    exit()

# Removing the 1st element of the embedding
# and setting emb1 and emb2 to their respective embedding

emb1 = np.array(embeds[0])[0][1:][np.newaxis, :]
emb2 = np.array(embeds[1])[0][1:][np.newaxis, :]

# Padding to ensure that the embedding vectors have the same size

shorter_emb = emb1 if emb1.shape[1] < emb2.shape[1] else emb2
longer_emb = emb2 if shorter_emb is emb1 else emb1
padding_width = ((0, 0), (0, longer_emb.shape[1] - shorter_emb.shape[1]), (0, 0))
padded_shorter_emb = np.pad(shorter_emb, padding_width, mode="constant")
assert padded_shorter_emb.shape == longer_emb.shape

similarity = cosine_similarity(
    longer_emb.reshape(1, -1), padded_shorter_emb.reshape(1, -1)
)
print(similarity)

model-00001-of-00002.safetensors:  20%|#9        | 1.98G/9.98G [00:00<?, ?B/s]

model-00002-of-00002.safetensors:   0%|          | 0.00/3.50G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk.


[[0.44371938]]


In [19]:
embeds = pipeline(
    [
        "soldier",
        "girl",
    ],
    tokenizer=tokenizer,
    eos_token_id=tokenizer.eos_token_id,
)

# Checking if either of the strings is empty
if len(embeds[0][0]) <= 1 or len(embeds[1][0]) <= 1:
    print("Not enough tokens to compare")
    exit()

# Removing the 1st element of the embedding
# and setting emb1 and emb2 to their respective embedding

emb1 = np.array(embeds[0])[0][1:][np.newaxis, :]
emb2 = np.array(embeds[1])[0][1:][np.newaxis, :]

# Padding to ensure that the embedding vectors have the same size

shorter_emb = emb1 if emb1.shape[1] < emb2.shape[1] else emb2
longer_emb = emb2 if shorter_emb is emb1 else emb1
padding_width = ((0, 0), (0, longer_emb.shape[1] - shorter_emb.shape[1]), (0, 0))
padded_shorter_emb = np.pad(shorter_emb, padding_width, mode="constant")
assert padded_shorter_emb.shape == longer_emb.shape

similarity = cosine_similarity(
    longer_emb.reshape(1, -1), padded_shorter_emb.reshape(1, -1)
)
print(similarity)

[[0.60990476]]
